In [1]:
import timm

from pathlib import Path
from fundus_data_toolkit.datamodules.classification import DDRDataModule
from fundus_data_toolkit.datamodules.classification import AptosDataModule
from fundus_data_toolkit.datasets.generic import get_image_dataset
from torch.utils.data import DataLoader
import torch
from tqdm.auto import tqdm
import pandas as pd

/home/clement/miniforge-pypy3/envs/dl/lib/python3.11/site-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()


In [2]:
model = timm.create_model(
    "convnext_base",
    num_classes=1,
    checkpoint_path="/home/clement/.cache/huggingface/hub/models--ClementP--FundusDRGrading-convnext_base/snapshots/23a24d5e721f0f93f3a21f57fc66c75d92efc69b/model.safetensors",
)
model = model.cuda().eval()

In [3]:
ddr_datamodule = DDRDataModule(
    data_dir="/home/clement/Documents/data/DDR-dataset/DR_grading/",
    img_size=1024,
    batch_size=32,
    num_workers=8,
)
ddr_datamodule = ddr_datamodule.setup_all()
aptos_datamodule = AptosDataModule(
    data_dir="/home/clement/Documents/data/aptos/",
    img_size=1024,
    batch_size=16,
    num_workers=4,
)
aptos_datamodule = aptos_datamodule.setup_all()
aptos_datamodule.train.return_indices = True

In [8]:
def infer_dataloader(dataloder):
    dataset = dataloader.dataset
    root_img = Path(dataloader.dataset.img_root[0])
    results = {"image_id": [], "prediction": []}
    for batch in tqdm(dataloader, total=len(dataloader)):
        indices = batch["index"]
        images = batch["image"].cuda()
        with torch.inference_mode():
            with torch.no_grad():
                predictions = model(images)

        for i, index in enumerate(indices):
            filepath = Path(dataset.img_filepath["image"][index])
            relative_path = filepath.relative_to(root_img)
            results["image_id"].append(relative_path)
            results["prediction"].append(predictions[i].item())
    return pd.DataFrame(results)


result_path = Path("CNN_results/DDR_test.pkl")
if result_path.exists():
    results_ddr = pd.read_pickle(result_path)
else:
    dataloader = ddr_datamodule.test_dataloader()
    df = infer_dataloader(dataloader)
    df.to_pickle(result_path)
    results_ddr = df


result_path = Path("CNN_results/APTOS.pkl")
if result_path.exists():
    results_aptos = pd.read_pickle(result_path)
else:
    dataloader = aptos_datamodule.train_dataloader()
    df = infer_dataloader(dataloader)
    df.to_pickle(result_path)
    results_aptos = df

result_path = Path("CNN_results/HMR.pkl")
if result_path.exists():
    results_hmr = pd.read_pickle(result_path)
else:
    results = {"image_id": [], "prediction": []}
    root_img = Path("/home/clement/Documents/data/IVisionHMR/output/fundus")
    dataset = get_image_dataset(root_img, (1536, 1536), precise_autocrop=False)
    dataset.return_indices = True
    dataloader = DataLoader(dataset, batch_size=16, shuffle=False, num_workers=8)
    for batch in tqdm(dataloader, total=len(dataloader)):
        indices = batch["index"]
        images = batch["image"].cuda()
        with torch.inference_mode():
            with torch.no_grad():
                predictions = model(images)

        for i, index in enumerate(indices):
            filepath = Path(dataset.img_filepath["image"][index])
            relative_path = filepath.relative_to(root_img)
            imageId, laterality = relative_path.parts[0], relative_path.parts[1]
            results["image_id"].append(f"{imageId}_{laterality}")
            results["prediction"].append(predictions[i].item())
    results_hmr = pd.DataFrame(results)

In [7]:
results_hmr.to_pickle(result_path)